# 02 — EDA: проверка гипотез

Этот notebook предназначен для **проектно-специфичных исследований**, которые не входят в автоматический технический профиль [[notebooks/02_eda.ipynb]]. Здесь проверяются связи target с отдельными признаками, взаимодействия признаков, корреляции и возможные направления feature engineering.

## Принцип исследования

Для каждого вопроса сохраняйте цепочку:

```text
Вопрос → метод → наблюдение → интерпретация → гипотеза → следующее действие
```

График без текстового вывода не считается завершённым исследованием. Корреляция показывает связь, но не доказывает причинность.

## Базовые шаблоны

### Target и категориальный признак

```python
FEATURE = "feature_name"
report = (
    train.groupby(FEATURE, dropna=False)[TARGET]
    .agg(rows="size", target_rate="mean")
    .sort_values("target_rate", ascending=False)
)
display(report)
sns.barplot(data=train, x=FEATURE, y=TARGET, errorbar=("ci", 95))
```

Проверяйте не только `target_rate`, но и размер каждой группы: высокая доля target на нескольких строках ненадёжна.

### Target и числовой признак

```python
FEATURE = "feature_name"
display(train.groupby(TARGET)[FEATURE].describe())
sns.boxplot(data=train, x=TARGET, y=FEATURE)
sns.histplot(data=train, x=FEATURE, hue=TARGET, stat="density", common_norm=False)
```

Смотрите форму распределения, выбросы, пропуски и возможную нелинейность. Средних значений недостаточно.

### Корреляции

```python
columns = [*numeric_features, TARGET]
correlation = train[columns].corr(method="spearman")
display(correlation)
sns.heatmap(correlation, annot=True, cmap="coolwarm", center=0)
```

- `pearson` — линейная связь;
- `spearman` — монотонная связь и меньшая чувствительность к выбросам;
- высокая корреляция между признаками может указывать на дублирование или multicollinearity;
- низкая корреляция с target не означает бесполезность нелинейного признака.

### Линейная зависимость двух числовых признаков

```python
sns.scatterplot(data=train, x="feature_x", y="feature_y", hue=TARGET)
sns.regplot(data=train, x="feature_x", y="feature_y", scatter=False)
```

Линия тренда — диагностический инструмент, а не доказательство, что линейная модель будет оптимальна.

### Взаимодействие двух категориальных признаков

```python
interaction = pd.pivot_table(
    train,
    values=TARGET,
    index="feature_a",
    columns="feature_b",
    aggfunc="mean",
)
display(interaction)
sns.heatmap(interaction, annot=True, cmap="viridis")
```

Взаимодействие полезно, когда эффект одного признака меняется внутри значений другого.

## Правила безопасности

- Не изменяйте файлы в `data/raw/`; работайте с `train.copy()`.
- Не используйте test target: его нет и он не должен появляться.
- Временные исследовательские колонки допустимы только внутри notebook.
- Принятую новую feature переносите в будущий `src/ml_project/features.py` и проверяйте отдельным экспериментом.
- Главные выводы переносите в [[docs/02_eda.md]], а идеи — в [[hypotheses/_index.md]].

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog, validate_feature_groups
import ml_project.config as project_config

# Перечитываем config.py без restart kernel.
project_config = importlib.reload(project_config)
DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET

catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
train = catalog.load(TRAIN_DATASET).copy()

if TARGET is None or TARGET not in train.columns:
    raise ValueError("Настройте TARGET в src/ml_project/config.py перед исследованием.")

feature_groups = validate_feature_groups(
    train,
    FEATURE_GROUPS,
    target=TARGET,
)
numeric_features = [
    *feature_groups["numeric"],
    *feature_groups["count"],
]
categorical_features = [
    *feature_groups["categorical"],
    *feature_groups["ordinal"],
]

FIGURE_DIR = PROJECT_ROOT / "assets" / "eda"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Train: {train.shape[0]} строк × {train.shape[1]} столбцов")
print(f"Target: {TARGET}")
print("Числовые + счётные:", ", ".join(numeric_features) or "—")
print("Категориальные + порядковые:", ", ".join(categorical_features) or "—")

## План исследования

Перед запуском графиков сформулируйте вопросы. Удаляйте нерелевантные пункты и добавляйте свои.

- [ ] Как каждый ключевой категориальный признак связан с target?
- [ ] Как распределения числовых признаков отличаются между классами target?
- [ ] Есть ли сильные Pearson/Spearman-корреляции?
- [ ] Есть ли признаки, дублирующие друг друга?
- [ ] Есть ли важные взаимодействия двух признаков?
- [ ] Какие наблюдения превращаются в проверяемые feature-гипотезы?

## 1. Target × категориальный признак

Измените `CATEGORICAL_FEATURE`. Для Titanic первым примером автоматически станет `Sex`, если столбец существует.

In [ ]:
CATEGORICAL_FEATURE = (
    "Sex"
    if "Sex" in categorical_features
    else (categorical_features[0] if categorical_features else None)
)

if CATEGORICAL_FEATURE is None:
    print("В FEATURE_GROUPS нет categorical/ordinal признаков.")
else:
    categorical_target_report = (
        train.groupby(CATEGORICAL_FEATURE, dropna=False)[TARGET]
        .agg(rows="size", target_rate="mean")
        .sort_values("target_rate", ascending=False)
    )
    display(categorical_target_report.style.format({"target_rate": "{:.2%}"}))

    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=train,
        x=CATEGORICAL_FEATURE,
        y=TARGET,
        errorbar=("ci", 95),
    )
    plt.title(f"{TARGET} по {CATEGORICAL_FEATURE}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### Вывод по категориальному признаку

- **Вопрос:**
- **Наблюдение:**
- **Надёжность:** достаточно ли строк в каждой группе?
- **Возможное объяснение:**
- **Feature-гипотеза:**
- **Следующий шаг:**

## 2. Target × числовой признак

Измените `NUMERIC_FEATURE`. Для Titanic первым примером автоматически станет `Age`, если столбец существует.

In [ ]:
NUMERIC_FEATURE = (
    "Age"
    if "Age" in numeric_features
    else (numeric_features[0] if numeric_features else None)
)

if NUMERIC_FEATURE is None:
    print("В FEATURE_GROUPS нет numeric/count признаков.")
else:
    display(train.groupby(TARGET)[NUMERIC_FEATURE].describe())

    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.boxplot(
        data=train,
        x=TARGET,
        y=NUMERIC_FEATURE,
        ax=axes[0],
    )
    sns.histplot(
        data=train,
        x=NUMERIC_FEATURE,
        hue=TARGET,
        stat="density",
        common_norm=False,
        kde=True,
        ax=axes[1],
    )
    figure.suptitle(f"Связь {NUMERIC_FEATURE} с {TARGET}")
    figure.tight_layout()
    plt.show()

### Вывод по числовому признаку

- **Вопрос:**
- **Наблюдение:**
- **Есть ли выбросы / пропуски / нелинейность:**
- **Возможное объяснение:**
- **Нужна ли трансформация или binning:**
- **Следующий шаг:**

## 3. Корреляции

Сравните `spearman` и `pearson`. Корреляционная матрица используется как диагностика, а не как автоматический способ отбора признаков.

In [ ]:
CORRELATION_METHOD = "spearman"  # замените на "pearson" для линейной связи
correlation_columns = list(dict.fromkeys([*numeric_features, TARGET]))

if len(correlation_columns) < 2:
    print("Недостаточно числовых признаков для корреляционного анализа.")
else:
    correlation_report = train[correlation_columns].corr(
        method=CORRELATION_METHOD
    )
    display(correlation_report)

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        correlation_report,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
    )
    plt.title(f"Корреляции ({CORRELATION_METHOD})")
    plt.tight_layout()
    plt.show()

### Вывод по корреляциям

- **Сильные связи с target:**
- **Сильные связи между признаками:**
- **Возможное дублирование / multicollinearity:**
- **Нелинейные связи, которые корреляция могла не увидеть:**
- **Следующий шаг:**

## 4. Линейная связь двух числовых признаков

Выберите `X_FEATURE` и `Y_FEATURE`. Цветом отмечается target, а линия показывает общий линейный тренд.

In [ ]:
X_FEATURE = numeric_features[0] if numeric_features else None
Y_FEATURE = numeric_features[1] if len(numeric_features) > 1 else None

if X_FEATURE is None or Y_FEATURE is None:
    print("Нужно минимум два numeric/count признака.")
else:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=train,
        x=X_FEATURE,
        y=Y_FEATURE,
        hue=TARGET,
        alpha=0.7,
    )
    sns.regplot(
        data=train,
        x=X_FEATURE,
        y=Y_FEATURE,
        scatter=False,
        color="black",
    )
    plt.title(f"{Y_FEATURE} относительно {X_FEATURE}")
    plt.tight_layout()
    plt.show()

### Вывод по линейной связи

- **Какая пара исследовалась:**
- **Видна ли линейная или монотонная связь:**
- **Различаются ли классы target:**
- **Есть ли кластеры или выбросы:**
- **Следующий шаг:**

## 5. Взаимодействие двух категориальных признаков

Для Titanic шаблон выберет `Sex × Pclass`, если оба признака настроены. Проверяйте также количество объектов в каждой комбинации.

In [ ]:
FIRST_FEATURE = (
    "Sex"
    if "Sex" in categorical_features
    else (categorical_features[0] if categorical_features else None)
)
SECOND_FEATURE = (
    "Pclass"
    if "Pclass" in categorical_features
    else (categorical_features[1] if len(categorical_features) > 1 else None)
)

if FIRST_FEATURE is None or SECOND_FEATURE is None:
    print("Нужно минимум два categorical/ordinal признака.")
else:
    interaction_rate = pd.pivot_table(
        train,
        values=TARGET,
        index=FIRST_FEATURE,
        columns=SECOND_FEATURE,
        aggfunc="mean",
    )
    interaction_count = pd.crosstab(
        train[FIRST_FEATURE],
        train[SECOND_FEATURE],
        dropna=False,
    )
    display(interaction_rate.style.format("{:.2%}"))
    display(interaction_count)

    plt.figure(figsize=(8, 5))
    sns.heatmap(
        interaction_rate,
        annot=True,
        fmt=".2f",
        cmap="viridis",
    )
    plt.title(f"{TARGET}: {FIRST_FEATURE} × {SECOND_FEATURE}")
    plt.tight_layout()
    plt.show()

### Вывод по взаимодействию

- **Какая пара исследовалась:**
- **Меняется ли эффект первого признака внутри второго:**
- **Достаточно ли наблюдений в комбинациях:**
- **Возможная interaction-feature:**
- **Следующий шаг:**

## Итог исследования

| Вопрос | Наблюдение | Надёжность | Решение | Ссылка на гипотезу |
|---|---|---|---|---|
|  |  |  |  |  |

### Кандидаты в признаки

| Feature | Источники | Тип | Почему может помочь | Как проверить |
|---|---|---|---|---|
|  |  |  |  |  |

### Завершение

- [ ] Каждый важный график имеет текстовый вывод.
- [ ] Проверен размер групп и устойчивость наблюдений.
- [ ] Корреляция не интерпретируется как причинность.
- [ ] Leakage-кандидаты отмечены отдельно.
- [ ] Значимые идеи оформлены в [[hypotheses/_index.md]].
- [ ] Главные выводы перенесены в [[docs/02_eda.md]].
- [ ] Принятые engineered features будут реализованы в `src/ml_project/features.py`, а не только в notebook.